In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [2]:
spark = SparkSession.builder \
    .appName("Semana12_Regresion_AutoTec_Jocelyn") \
    .getOrCreate()

ruta_datos = "/home/jovyan/work/autotec/Jocelyn/datos_etiquetados_kmeans"

df_clusters = spark.read.parquet(ruta_datos)

print("Datos cargados:", df_clusters.count())
df_clusters.printSchema()
df_clusters.show(5, truncate=False)

Datos cargados: 1904
root
 |-- precio_num: double (nullable = true)
 |-- km_num: double (nullable = true)
 |-- year_num: double (nullable = true)
 |-- marca_cat: double (nullable = true)
 |-- combustible_cat: double (nullable = true)
 |-- features: vector (nullable = true)
 |-- scaledFeatures: vector (nullable = true)
 |-- prediction: integer (nullable = true)

+----------+---------+--------+---------+---------------+-----------------------------------+----------------------------------------------------------------------------------+----------+
|precio_num|km_num   |year_num|marca_cat|combustible_cat|features                           |scaledFeatures                                                                    |prediction|
+----------+---------+--------+---------+---------------+-----------------------------------+----------------------------------------------------------------------------------+----------+
|22997.0   |272940.0 |2024.0  |16.0     |0.0            |[22997.0,272940

In [3]:
df_reg = df_clusters.select(
    col("precio_num").cast("double").alias("label"),
    col("km_num").cast("double"),
    col("year_num").cast("double"),
    col("marca_cat").cast("double"),
    col("combustible_cat").cast("double")
).dropna()

assembler = VectorAssembler(
    inputCols=["km_num", "year_num", "marca_cat", "combustible_cat"],
    outputCol="features"
)

df_vector = assembler.transform(df_reg)

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures"
)

scaler_model = scaler.fit(df_vector)
df_scaled = scaler_model.transform(df_vector)

train_reg, test_reg = df_scaled.randomSplit([0.7, 0.3], seed=42)

print("Entrenamiento:", train_reg.count())
print("Prueba:", test_reg.count())

Entrenamiento: 1382
Prueba: 522


In [4]:
lr_reg = LinearRegression(
    featuresCol="scaledFeatures",
    labelCol="label",
    maxIter=20
)

lr_model = lr_reg.fit(train_reg)

pred_lr = lr_model.transform(test_reg)

pred_lr.select("label", "prediction", "km_num", "year_num").show(20, truncate=False)

+------+--------------------+---------+--------+
|label |prediction          |km_num   |year_num|
+------+--------------------+---------+--------+
|117.0 |3.073588516997385E7 |905680.0 |2018.0  |
|127.0 |3.3007235038420677E7|1860000.0|2017.0  |
|137.0 |1.7329798313766003E7|650000.0 |2019.0  |
|137.0 |4.0803129765190125E7|1920000.0|2018.0  |
|147.0 |5.338220471969223E7 |3000000.0|2017.0  |
|157.0 |2.1985036099039555E7|406000.0 |2020.0  |
|157.0 |8931525.686625004   |545590.0 |2023.0  |
|187.0 |2.429179744371128E7 |828000.0 |2021.0  |
|197.0 |1.9902037404021263E7|924130.0 |2019.0  |
|207.0 |2.1202521566303253E7|507770.0 |2021.0  |
|217.0 |1.1087582760757923E7|208190.0 |2020.0  |
|257.0 |4.259962355828953E7 |1200000.0|2013.0  |
|267.0 |5494253.987533569   |200000.0 |2023.0  |
|267.0 |3.358379616339111E7 |1006220.0|2017.0  |
|277.0 |2.2694602122763634E7|850000.0 |2022.0  |
|297.0 |6343597.516117573   |310000.0 |2023.0  |
|297.0 |1.246480710563469E7 |1100000.0|2024.0  |
|397.0 |2.7050528316

In [5]:
evaluator_r2 = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="r2"
)

evaluator_rmse = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

r2_lr = evaluator_r2.evaluate(pred_lr)
rmse_lr = evaluator_rmse.evaluate(pred_lr)

print("RESULTADOS REGRESIÓN LINEAL")
print(f"R2: {r2_lr:.4f}")
print(f"RMSE: {rmse_lr:.2f}")

RESULTADOS REGRESIÓN LINEAL
R2: 0.0092
RMSE: 37303095.10


In [6]:
rf_reg = RandomForestRegressor(
    featuresCol="scaledFeatures",
    labelCol="label",
    numTrees=30,
    maxDepth=5,
    seed=42
)

rf_reg_model = rf_reg.fit(train_reg)

pred_rf = rf_reg_model.transform(test_reg)

r2_rf = evaluator_r2.evaluate(pred_rf)
rmse_rf = evaluator_rmse.evaluate(pred_rf)

print("RESULTADOS RANDOM FOREST REGRESSOR")
print(f"R2: {r2_rf:.4f}")
print(f"RMSE: {rmse_rf:.2f}")

RESULTADOS RANDOM FOREST REGRESSOR
R2: 0.0889
RMSE: 35771679.69


In [7]:
print("COMPARACIÓN MODELOS DE REGRESIÓN")
print(f"Regresión Lineal - R2: {r2_lr:.4f} | RMSE: {rmse_lr:.2f}")
print(f"Random Forest    - R2: {r2_rf:.4f} | RMSE: {rmse_rf:.2f}")

print("""
Interpretación:
En esta etapa se entrenaron modelos supervisados para predecir el precio de los vehículos.
Se usaron variables como kilometraje, año, marca y combustible.
El R2 indica qué tan bien el modelo explica el precio, mientras que el RMSE muestra el error promedio de predicción.
""")

COMPARACIÓN MODELOS DE REGRESIÓN
Regresión Lineal - R2: 0.0092 | RMSE: 37303095.10
Random Forest    - R2: 0.0889 | RMSE: 35771679.69

Interpretación:
En esta etapa se entrenaron modelos supervisados para predecir el precio de los vehículos.
Se usaron variables como kilometraje, año, marca y combustible.
El R2 indica qué tan bien el modelo explica el precio, mientras que el RMSE muestra el error promedio de predicción.

